# Experiment 1 v2 — K-Shot Ablation (Multi-Model, Balanced Prompt)

**Perubahan dari v1:**
- Prompt v2 (balanced, tidak bias ke pneumothorax)
- Support 5 model dari 3 arsitektur berbeda
- Pilih model di Cell 2 → jalankan notebook 1 model per run
- Checkpoint terpisah per model

**Model yang didukung:**

| ID | Model | Architecture | Backend |
|----|-------|--------------|---------|
| M1 | NV-Reason-CXR-3B | Qwen2.5-VL | qwen |
| M2 | ChestXReasoner | Qwen2.5-VL | qwen |
| M3 | Qwen2.5-VL-3B-Instruct | Qwen2.5-VL | qwen |
| M4 | MedGemma-4B-IT | Gemma 3 | gemma |
| M5 | Mini-InternVL2-4B-Medical | InternLM2 | internvl |

## Cell 1 — Imports & Setup

In [1]:
import sys
import json
import yaml
import re
import time
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from PIL import Image
import os

import torch

PROJECT_ROOT = Path("/home/equity/")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

try:
    sys.stdout.reconfigure(encoding="utf-8")
except AttributeError:
    pass

print(f"Working dir : {os.getcwd()}")
print(f"Python      : {sys.executable}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Working dir : /home/equity
Python      : /usr/bin/python
PyTorch     : 2.10.0a0+a36e1d39eb.nv26.01.42222806
CUDA        : True
GPU         : NVIDIA GeForce RTX 4090
VRAM        : 25.3 GB


## Cell 2 — Pilih Model

**Ganti `ACTIVE_MODEL` setiap kali mau run model berbeda.** Jalankan seluruh notebook dari awal setiap ganti model.

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  GANTI DI SINI — pilih salah satu: "M1", "M2", ... "M5"   ║
# ╚══════════════════════════════════════════════════════════════╝
ACTIVE_MODEL = "M3"


MODEL_REGISTRY = {
    "M1": {
        "name":       "NV-Reason-CXR-3B",
        "path":       "./models/NV-Reason-CXR-3B",
        "backend":    "qwen",
        "short_id":   "nv_reason",
    },
    "M2": {
        "name":       "ChestXReasoner",
        "path":       "./models/ChestXReasoner",
        "backend":    "qwen",
        "short_id":   "chestxreasoner",
    },
    "M3": {
        "name":       "Qwen2.5-VL-3B-Instruct",
        "path":       "./models/Qwen2.5-VL-3B-Instruct",
        "backend":    "qwen",
        "short_id":   "qwen25vl_base",
    },
    "M4": {
        "name":       "MedGemma-4B-IT",
        "path":       "./models/MedGemma-4B-IT",
        "backend":    "gemma",
        "short_id":   "medgemma",
    },
    "M5": {
        "name":       "Mini-InternVL2-4B-Medical",
        "path":       "./models/Mini-InternVL2-4B-Medical",
        "backend":    "internvl",
        "short_id":   "internvl_med",
    },
}

M = MODEL_REGISTRY[ACTIVE_MODEL]
MODEL_NAME    = M["name"]
MODEL_PATH    = M["path"]
MODEL_BACKEND = M["backend"]
MODEL_SHORT   = M["short_id"]

print(f"\n{'='*60}")
print(f"  ACTIVE MODEL: [{ACTIVE_MODEL}] {MODEL_NAME}")
print(f"  Backend     : {MODEL_BACKEND}")
print(f"  Path        : {MODEL_PATH}")
print(f"{'='*60}")


  ACTIVE MODEL: [M3] Qwen2.5-VL-3B-Instruct
  Backend     : qwen
  Path        : ./models/Qwen2.5-VL-3B-Instruct


## Cell 3 — Load Config & Paths

In [3]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment_config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

DATA_ROOT  = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "results" / f"exp1v2_{MODEL_SHORT}"
CKPT_PATH  = OUTPUT_DIR / "checkpoint.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_SHOTS    = cfg["experiments"]["k_shots"]          # [0, 1, 2, 4, 8]
NUM_SEEDS  = cfg["experiments"]["num_random_seeds"]  # 5
SEEDS      = list(range(NUM_SEEDS))

# Generation parameters (kontrol — sama untuk semua model)
GEN_PARAMS = {
    "max_new_tokens": cfg["model"]["max_new_tokens"],  # 1024
    "temperature":    cfg["model"]["temperature"],      # 0.1
    "top_p":          cfg["model"]["top_p"],            # 0.9
}

print(f"k_shots     : {K_SHOTS}")
print(f"seeds       : {SEEDS}")
print(f"output_dir  : {OUTPUT_DIR}")
print(f"checkpoint  : {CKPT_PATH}")
print(f"gen_params  : {GEN_PARAMS}")

k_shots     : [0, 1, 2, 4, 8]
seeds       : [0, 1, 2, 3, 4]
output_dir  : /home/equity/results/exp1v2_qwen25vl_base
checkpoint  : /home/equity/results/exp1v2_qwen25vl_base/checkpoint.jsonl
gen_params  : {'max_new_tokens': 1024, 'temperature': 0.1, 'top_p': 0.9}


## Cell 4 — Prompt v2 (Balanced)

In [4]:
# Import prompt v2 dari src/prompt_builder.py (single source of truth)
from src.prompt_builder import (
    SYSTEM_PROMPT_V2,
    USER_QUERY_V2,
    build_few_shot_context_v2 as build_few_shot_context,
    _load_image,
)

print("Prompt v2 loaded from src/prompt_builder.py")
print(f"System prompt preview: {SYSTEM_PROMPT_V2[:80]}...")
print(f"User query: {USER_QUERY_V2}")

Prompt v2 loaded from src/prompt_builder.py
System prompt preview: You are an expert radiologist assistant specialized in chest X-ray (CXR) analysi...
User query: Analyze this chest X-ray.


## Cell 5 — Multi-Backend Engine

In [5]:
# Import MultiModelEngine dari src/inference.py (single source of truth)
from src.inference import MultiModelEngine

print("MultiModelEngine loaded from src/inference.py")

MultiModelEngine loaded from src/inference.py


## Cell 6 — Load Test Sets

In [6]:
def _img_path_chexpert_test(file_id: str, label_str: str) -> Path:
    subfolder = "internal_test_pneumothorax" if label_str == "pneumothorax" else "internal_test_normal"
    return DATA_ROOT / "chexpert_plus" / "images" / subfolder / f"{file_id}.png"

def _img_path_rsunair(image_id: str, label_int: int) -> Path:
    subfolder = "pneumothorax" if label_int == 1 else "normal"
    return DATA_ROOT / "rs_unair" / "images" / subfolder / f"{image_id}.png"


# ── CheXpert Plus Internal Test ────────────────────────────────────────
chex_df = pd.read_csv(cfg["data"]["chexpert_internal_test_csv"], encoding="utf-8-sig")
chex_df["label_int"] = chex_df["label"].map({"pneumothorax": 1, "normal": 0})
chex_df["image_id"]  = chex_df["file_id"]
chex_images = [
    _img_path_chexpert_test(row["file_id"], row["label"])
    for _, row in chex_df.iterrows()
]
missing_chex = sum(1 for p in chex_images if not p.exists())
print(f"CheXpert Internal : {len(chex_df)} images  |  missing: {missing_chex}")
print(f"  PTX={chex_df['label_int'].sum()}  Normal={len(chex_df)-chex_df['label_int'].sum()}")


# ── RS UNAIR External ──────────────────────────────────────────────────
rs_labels = pd.read_csv(cfg["data"]["rsunair_labels_csv"], encoding="utf-8-sig")
rs_meta   = pd.read_csv(
    DATA_ROOT / "rs_unair" / "metadata_normalized_dicom.csv",
    encoding="utf-8-sig", sep=None, engine="python"
)
rs_labels_renamed  = rs_labels.rename(columns={"image_id": "file_id", "label": "label_int"})
rs_meta_merged     = rs_meta.merge(rs_labels_renamed, on="file_id", how="inner")
rs_meta_merged["image_id"] = rs_meta_merged["file_id"]

rs_images = [
    _img_path_rsunair(row["image_id"], row["label_int"])
    for _, row in rs_meta_merged.iterrows()
]
missing_rs = sum(1 for p in rs_images if not p.exists())
print(f"RS UNAIR External : {len(rs_meta_merged)} images  |  missing: {missing_rs}")
print(f"  PTX={rs_meta_merged['label_int'].sum()}  Normal={len(rs_meta_merged)-rs_meta_merged['label_int'].sum()}")


TEST_SETS = {
    "chexpert_internal": {
        "name"   : "CheXpert Plus (Internal)",
        "rows"   : chex_df.reset_index(drop=True),
        "images" : chex_images,
    },
    "rsunair_external": {
        "name"   : "RS UNAIR (External)",
        "rows"   : rs_meta_merged.reset_index(drop=True),
        "images" : rs_images,
    },
}

print("\nTest sets loaded.")

CheXpert Internal : 200 images  |  missing: 0
  PTX=100  Normal=100
RS UNAIR External : 278 images  |  missing: 0
  PTX=173  Normal=105

Test sets loaded.


## Cell 7 — Load Model

In [7]:
from transformers import Qwen2_5_VLForConditionalGeneration
import os

model_path = "/home/equity/models/Qwen2.5-VL-3B-Instruct"

try:
    # Memuat hanya konfigurasi dan memeriksa struktur bobot
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_path, 
        device_map="meta", # Tidak menggunakan VRAM, hanya cek struktur
        check_checkpoint=True
    )
    print("\n✅ VERIFIKASI SUKSES: Semua kunci (keys) ditemukan dalam checkpoint.")
except Exception as e:
    print(f"\n❌ VERIFIKASI GAGAL: Terdeteksi masalah pada file model.\nError: {e}")


❌ VERIFIKASI GAGAL: Terdeteksi masalah pada file model.
Error: Qwen2_5_VLForConditionalGeneration.__init__() got an unexpected keyword argument 'check_checkpoint'


In [8]:
engine = MultiModelEngine(
    model_path=MODEL_PATH,
    backend=MODEL_BACKEND,
    quantize_4bit=True,
)

print(f"\nModel [{ACTIVE_MODEL}] {MODEL_NAME} siap.")
print(f"VRAM used: {engine.vram_used_gb():.2f} GB")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[Engine] Loading: ./models/Qwen2.5-VL-3B-Instruct (backend=qwen, 4bit=True)


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

[Engine] Ready. VRAM: 2.43 GB

Model [M3] Qwen2.5-VL-3B-Instruct siap.
VRAM used: 2.43 GB


## Cell 8 — Sanity Check: Single Inference

In [9]:
# Test dengan 1 image (zero-shot)
test_img_path = chex_images[0]
test_img = _load_image(test_img_path)

print(f"Testing single inference on: {test_img_path.name}")

text_out, t_out = engine.generate(
    system_prompt=SYSTEM_PROMPT_V2,
    user_text="",  # zero-shot
    image=test_img,
    **GEN_PARAMS,
)

print(f"Time: {t_out:.1f}s")
print(f"Output length: {len(text_out)} chars")
print(f"Preview: {text_out[:300]}...")
print(f"\nSanity check PASSED.")

Testing single inference on: patient49816_study1_view1_frontal.png
Time: 41.6s
Output length: 1072 chars
Preview: **SYSTEMATIC REVIEW:**
- **Lungs:** The lung fields appear clear without any signs of consolidation, effusion, or infiltrates.
- **Heart:** The cardiac silhouette is within normal limits, and there are no signs of cardiomegaly or pericardial effusion.
- **Mediastinum:** The mediastinal structures ar...

Sanity check PASSED.


## Cell 9 — Init Few-Shot Selector

In [11]:
from src.example_selector import FewShotSelector

selector = FewShotSelector(
    positive_csv="data/chexpert_ptx_positive.csv",
    negative_csv="data/chexpert_ptx_negative.csv",
)

[FewShotSelector] Pool loaded — PTX: 150, Normal: 150


## Cell 10 — Checkpoint Helper

In [12]:
def load_done_keys(ckpt_path: Path) -> set:
    done = set()
    if ckpt_path.exists():
        with open(ckpt_path, encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                    done.add((r["k_shot"], r["strategy"], r["seed"],
                              r["test_set"], r["image_id"]))
                except Exception:
                    pass
    return done


def append_jsonl(path: Path, records: list[dict]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


done_keys = load_done_keys(CKPT_PATH)
print(f"Records sudah selesai (resume): {len(done_keys)}")

total_target = len(K_SHOTS) * len(SEEDS) * sum(len(ts["images"]) for ts in TEST_SETS.values())
print(f"Total target inferences        : {total_target}")
print(f"Sisa yang harus dijalankan     : {total_target - len(done_keys)}")

Records sudah selesai (resume): 0
Total target inferences        : 11950
Sisa yang harus dijalankan     : 11950


## Cell 11 — Run Experiment 1 v2

> **Estimasi:** ~11,950 inferences per model. Dengan ~10-12s per inference = **~33-40 jam**.
>
> Cell ini **aman di-restart** — progress tersimpan otomatis per image. Jika kernel crash, restart dari Cell 7 lalu jalankan Cell 11 lagi.

In [13]:
print("=" * 65)
print(f"EXPERIMENT 1 v2: K-Shot Ablation")
print(f"Model: [{ACTIVE_MODEL}] {MODEL_NAME}")
print(f"Prompt: v2 (balanced)")
print("=" * 65)

exp_start = datetime.now()

for k in K_SHOTS:
    for seed in SEEDS:
        # ── Select few-shot examples ────────────────────────────────
        examples = selector.select("random", k=k, seed=seed)

        # ── Build few-shot context text ─────────────────────────────
        fewshot_text = build_few_shot_context(examples, k_shot=k)

        # ── Run on BOTH test sets ───────────────────────────────────
        for ts_key, test_set in TEST_SETS.items():
            ts_name = test_set["name"]

            # Filter belum selesai
            todo = [
                (img_path, row)
                for img_path, (_, row) in zip(test_set["images"], test_set["rows"].iterrows())
                if (k, "random", seed, ts_name, str(row["image_id"])) not in done_keys
                and img_path.exists()
            ]

            print(f"\n[k={k} seed={seed}] {ts_name}")

            if not todo:
                print("  \u2192 sudah selesai, skip.")
                continue

            print(f"  \u2192 {len(todo)} images to process...")
            batch_records = []

            for idx, (img_path, row) in enumerate(todo, 1):
                image_id = str(row["image_id"])
                print(f"    [{idx}/{len(todo)}] {image_id}", end=" ", flush=True)

                # Load image
                img = _load_image(img_path)

                try:
                    generated_text, t = engine.generate(
                        system_prompt=SYSTEM_PROMPT_V2,
                        user_text=fewshot_text,
                        image=img,
                        **GEN_PARAMS,
                    )
                    print(f"({t:.1f}s) \u2713")
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    gc.collect()
                    generated_text, t = "", 0.0
                    print("OOM \u2717")
                except Exception as e:
                    generated_text, t = "", 0.0
                    print(f"ERROR: {e} \u2717")

                record = {
                    "experiment_type" : "k_shot_ablation",
                    "model_id"        : ACTIVE_MODEL,
                    "model_name"      : MODEL_NAME,
                    "prompt_version"  : "v2",
                    "k_shot"          : k,
                    "strategy"        : "random",
                    "seed"            : seed,
                    "test_set"        : ts_name,
                    "image_id"        : image_id,
                    "ground_truth"    : int(row["label_int"]),
                    "generated_text"  : generated_text,
                    "inference_time"  : round(t, 3),
                    "timestamp"       : datetime.now().isoformat(),
                }
                batch_records.append(record)
                done_keys.add((k, "random", seed, ts_name, image_id))

                # Flush setiap 10 records
                if len(batch_records) % 10 == 0:
                    append_jsonl(CKPT_PATH, batch_records)
                    batch_records = []

                # Close image to free memory
                img.close()

            # Flush sisa
            if batch_records:
                append_jsonl(CKPT_PATH, batch_records)

            print(f"  \u2192 checkpoint saved.")

elapsed = (datetime.now() - exp_start).total_seconds() / 3600
print(f"\n{'=' * 65}")
print(f"[{ACTIVE_MODEL}] {MODEL_NAME} — Experiment 1 v2 selesai.")
print(f"Total waktu sesi ini : {elapsed:.2f} jam")
print(f"Total records        : {len(done_keys)}")

EXPERIMENT 1 v2: K-Shot Ablation
Model: [M3] Qwen2.5-VL-3B-Instruct
Prompt: v2 (balanced)

[k=0 seed=0] CheXpert Plus (Internal)
  → 200 images to process...
    [1/200] patient49816_study1_view1_frontal (40.6s) ✓
    [2/200] patient51966_study2_view1_frontal (8.7s) ✓
    [3/200] patient10406_study1_view1_frontal (34.5s) ✓
    [4/200] patient39377_study1_view1_frontal (28.6s) ✓
    [5/200] patient10464_study1_view2_frontal (32.7s) ✓
    [6/200] patient07182_study6_view1_frontal (29.2s) ✓
    [7/200] patient01124_study4_view1_frontal (39.9s) ✓
    [8/200] patient15726_study2_view1_frontal (6.5s) ✓
    [9/200] patient35607_study7_view1_frontal (40.0s) ✓
    [10/200] patient39232_study2_view1_frontal (6.5s) ✓
    [11/200] patient11210_study5_view1_frontal (8.6s) ✓
    [12/200] patient23439_study5_view1_frontal (35.2s) ✓
    [13/200] patient20200_study12_view1_frontal (34.9s) ✓
    [14/200] patient60146_study6_view1_frontal (49.5s) ✓
    [15/200] patient49769_study2_view1_frontal (37.9s) ✓

## Cell 12 — Quick Evaluation

In [14]:
from src.evaluate import extract_diagnosis, compute_classification_metrics

# Load checkpoint
records = []
if CKPT_PATH.exists():
    with open(CKPT_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                records.append(json.loads(line.strip()))
            except Exception:
                pass

print(f"Total records: {len(records)}")

df = pd.DataFrame(records)
df["predicted"] = df["generated_text"].apply(extract_diagnosis)

print(f"Invalid predictions: {(df['predicted'] == -1).sum()}")
print(f"k_shots: {sorted(df['k_shot'].unique())}")
print(f"test_sets: {df['test_set'].unique().tolist()}")

Total records: 11950
Invalid predictions: 0
k_shots: [np.int64(0), np.int64(1), np.int64(2), np.int64(4), np.int64(8)]
test_sets: ['CheXpert Plus (Internal)', 'RS UNAIR (External)']


In [15]:
# Quick summary: mean ± std across seeds
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

rows = []
for (k, ts), grp in df.groupby(["k_shot", "test_set"]):
    seed_metrics = []
    for seed in grp["seed"].unique():
        sg = grp[grp["seed"] == seed]
        m = compute_classification_metrics(sg["ground_truth"].tolist(), sg["predicted"].tolist())
        seed_metrics.append(m)
    sm_df = pd.DataFrame(seed_metrics)
    rows.append({
        "k_shot": k, "test_set": ts,
        "acc":  f"{sm_df['accuracy'].mean():.4f}\u00b1{sm_df['accuracy'].std():.4f}",
        "sens": f"{sm_df['sensitivity'].mean():.4f}\u00b1{sm_df['sensitivity'].std():.4f}",
        "spec": f"{sm_df['specificity'].mean():.4f}\u00b1{sm_df['specificity'].std():.4f}",
        "f1":   f"{sm_df['f1'].mean():.4f}\u00b1{sm_df['f1'].std():.4f}",
    })

result_df = pd.DataFrame(rows)

print(f"\n{'='*70}")
print(f"  [{ACTIVE_MODEL}] {MODEL_NAME} — Quick Summary (Prompt v2)")
print(f"{'='*70}")

for ts in result_df["test_set"].unique():
    print(f"\n  {ts}:")
    sub = result_df[result_df["test_set"] == ts]
    for _, r in sub.iterrows():
        print(f"    k={int(r['k_shot']):d}  Acc={r['acc']}  F1={r['f1']}  Sens={r['sens']}  Spec={r['spec']}")


  [M3] Qwen2.5-VL-3B-Instruct — Quick Summary (Prompt v2)

  CheXpert Plus (Internal):
    k=0  Acc=0.5640±0.0042  F1=0.6432±0.0037  Sens=0.7860±0.0055  Spec=0.3420±0.0045
    k=1  Acc=0.5320±0.0175  F1=0.6166±0.0283  Sens=0.7560±0.0713  Spec=0.3080±0.0581
    k=2  Acc=0.5020±0.0202  F1=0.6139±0.0264  Sens=0.7940±0.0577  Spec=0.2100±0.0292
    k=4  Acc=0.4900±0.0112  F1=0.6075±0.0304  Sens=0.7940±0.0844  Spec=0.1860±0.0654
    k=8  Acc=0.4780±0.0110  F1=0.5990±0.0106  Sens=0.7800±0.0255  Spec=0.1760±0.0279

  RS UNAIR (External):
    k=0  Acc=0.6180±0.0016  F1=0.7639±0.0012  Sens=0.9930±0.0026  Spec=0.0000±0.0000
    k=1  Acc=0.6122±0.0078  F1=0.7595±0.0060  Sens=0.9838±0.0125  Spec=0.0000±0.0000
    k=2  Acc=0.6108±0.0047  F1=0.7584±0.0036  Sens=0.9815±0.0075  Spec=0.0000±0.0000
    k=4  Acc=0.6115±0.0076  F1=0.7589±0.0058  Sens=0.9827±0.0123  Spec=0.0000±0.0000
    k=8  Acc=0.6151±0.0000  F1=0.7617±0.0000  Sens=0.9884±0.0000  Spec=0.0000±0.0000


## Cell 13 — Simpan Deduplicated Results

In [16]:
seen = set()
unique_records = []
for r in records:
    key = (r["k_shot"], r["strategy"], r["seed"], r["test_set"], r["image_id"])
    if key not in seen:
        seen.add(key)
        unique_records.append(r)

final_path = OUTPUT_DIR / "all_results.jsonl"
with open(final_path, "w", encoding="utf-8") as f:
    for r in unique_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved: {final_path}")
print(f"Records: {len(unique_records)} unique")

Saved: /home/equity/results/exp1v2_qwen25vl_base/all_results.jsonl
Records: 11950 unique


## Cell 14 — Bias Metrics (PBR & SDR)

In [17]:
print(f"\n{'='*70}")
print(f"  BIAS ANALYSIS — [{ACTIVE_MODEL}] {MODEL_NAME}")
print(f"{'='*70}")

for ts in df["test_set"].unique():
    ts_df = df[df["test_set"] == ts]
    print(f"\n  {ts}:")
    print(f"  {'k':>3} | {'PBR':>6} | {'Spec':>6} | {'SDR':>7} | {'Sens':>6}")
    print(f"  {'-'*40}")

    spec_k0 = None

    for k in sorted(ts_df["k_shot"].unique()):
        k_df = ts_df[ts_df["k_shot"] == k]
        n_actual_pos = (k_df["ground_truth"] == 1).sum()
        n_pred_pos   = (k_df["predicted"] == 1).sum()

        pbr = n_pred_pos / n_actual_pos if n_actual_pos > 0 else 0

        tp = ((k_df["predicted"] == 1) & (k_df["ground_truth"] == 1)).sum()
        tn = ((k_df["predicted"] == 0) & (k_df["ground_truth"] == 0)).sum()
        fp = ((k_df["predicted"] == 1) & (k_df["ground_truth"] == 0)).sum()
        fn = ((k_df["predicted"] == 0) & (k_df["ground_truth"] == 1)).sum()

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0

        if k == 0:
            spec_k0 = spec
            sdr = 0.0
        else:
            sdr = ((spec_k0 - spec) / spec_k0 * 100) if spec_k0 and spec_k0 > 0 else 0

        print(f"  {k:3d} | {pbr:6.2f} | {spec:6.3f} | {sdr:6.1f}% | {sens:6.3f}")


  BIAS ANALYSIS — [M3] Qwen2.5-VL-3B-Instruct

  CheXpert Plus (Internal):
    k |    PBR |   Spec |     SDR |   Sens
  ----------------------------------------
    0 |   1.44 |  0.342 |    0.0% |  0.786
    1 |   1.45 |  0.308 |    9.9% |  0.756
    2 |   1.58 |  0.210 |   38.6% |  0.794
    4 |   1.61 |  0.186 |   45.6% |  0.794
    8 |   1.60 |  0.176 |   48.5% |  0.780

  RS UNAIR (External):
    k |    PBR |   Spec |     SDR |   Sens
  ----------------------------------------
    0 |   1.60 |  0.000 |    0.0% |  0.993
    1 |   1.59 |  0.000 |    0.0% |  0.984
    2 |   1.59 |  0.000 |    0.0% |  0.982
    4 |   1.59 |  0.000 |    0.0% |  0.983
    8 |   1.60 |  0.000 |    0.0% |  0.988


## Cell 15 — Next Steps

Setelah model ini selesai:

1. **Ganti `ACTIVE_MODEL`** di Cell 2 ke model berikutnya
2. **Restart kernel** → jalankan seluruh notebook dari awal
3. Ulangi sampai semua 5 model selesai
4. Hasil tersimpan di folder terpisah per model:
   ```
   results/exp1v2_nv_reason/all_results.jsonl
   results/exp1v2_chestxreasoner/all_results.jsonl
   results/exp1v2_qwen25vl_base/all_results.jsonl
   results/exp1v2_medgemma/all_results.jsonl
   results/exp1v2_internvl_med/all_results.jsonl
   ```
5. Setelah semua selesai → jalankan notebook **analisis cross-model**

In [18]:
# Status check: model mana saja yang sudah ada hasilnya?
results_root = PROJECT_ROOT / "results"

print("\nStatus semua model:")
for mid, info in MODEL_REGISTRY.items():
    result_dir = results_root / f"exp1v2_{info['short_id']}"
    result_file = result_dir / "all_results.jsonl"
    if result_file.exists():
        n_lines = sum(1 for _ in open(result_file))
        status = f"DONE ({n_lines} records)"
    elif (result_dir / "checkpoint.jsonl").exists():
        n_lines = sum(1 for _ in open(result_dir / "checkpoint.jsonl"))
        status = f"IN PROGRESS ({n_lines} records)"
    else:
        status = "NOT STARTED"

    marker = "\u2714" if "DONE" in status else "\u25cb" if "PROGRESS" in status else "\u2717"
    current = " \u2190 CURRENT" if mid == ACTIVE_MODEL else ""
    print(f"  {marker} [{mid}] {info['name']:30s} {status}{current}")


Status semua model:
  ✗ [M1] NV-Reason-CXR-3B               NOT STARTED
  ✗ [M2] ChestXReasoner                 NOT STARTED
  ✔ [M3] Qwen2.5-VL-3B-Instruct         DONE (11950 records) ← CURRENT
  ✗ [M4] MedGemma-4B-IT                 NOT STARTED
  ✗ [M5] Mini-InternVL2-4B-Medical      NOT STARTED
